### imports & set up

In [53]:
import os
import sys
import json
import torch
import functools
import numpy as np
import pandas as pd
import gc
from typing import List
from torch import Tensor
from transformers import AutoTokenizer
from jaxtyping import Int
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import scipy.stats as stats
from scipy.stats import mannwhitneyu, norm
import dotenv
import os
from pathlib import Path
import shutil
import traceback
from huggingface_hub.constants import HF_HUB_CACHE
from transformer_lens.model_bridge import TransformerBridge as TransformerBridge
from pathlib import Path

In [54]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [55]:
print(torch.__version__) #should be 2.6 

2.6.0+cu124


In [56]:
# #load environment variables from .env file
# if not dotenv.load_dotenv():
#     print("Could not find dotenv file...")
# else:
#     print("Successfully loaded dotenv file. ", end="")
#     print(f"Environment PROJECT={os.environ.get('PROJECT')}")

#test

In [57]:
#set up paths
src_path = Path().resolve() / "../src"
sys.path.append(src_path)
project_root = src_path.resolve().parent  
sys.path.append(str(project_root))

In [58]:
#local imports from src 
import src.interp_toolkit as interp_toolkit

In [59]:
# known issue with statsmodels means there is an issue with the following package 
#https://github.com/statsmodels/statsmodels/issues/9584
# import statsmodels.api as sm
# from statsmodels.formula.api import ols

In [60]:
torch.set_grad_enabled(False)


In [61]:
interp_toolkit.load_environment_variables()

Successfully loaded dotenv file. Environment PROJECT=my_project


In [62]:
interp_toolkit.check_environment()

Hugging face cache inside the dockerfile lives at /home/aleshins/.cache
Hugging face token is hf_...Xso


### basic demo

##### load a model by name

In [69]:

#set its familty as a global variable -- this is only for purposes of prompt formatting
MODEL_FAMILY = "qwen" #options: "qwen", "llama", "mistral", "gemma"

#load a model 
model = interp_toolkit.load_model("Qwen/Qwen3.5-2B",device="cuda:1", model_family=MODEL_FAMILY)
    



Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Moving model to device:  cuda
Model 'Qwen/Qwen3.5-2B' loaded successfully on cuda:1.


##### format a prompt using a chat template

In [64]:
#TODO: write function to read in stimuli and prompts from json files later--this is a simple demo for now
#suppose this might be constructed from a list of stimuli read in from a file.


sample_prompt= 'What is the capital city of France?'
system_message = "You are a helpful assistant."

#format a prompt for this stimulus item
this_prompt = interp_toolkit.create_prompt(MODEL_FAMILY, system_message, sample_prompt)
print(this_prompt)

<|im_start|>system
You are a helpful assistant..<|im_end|>
<|im_start|>user
What is the capital city of France?<|im_end|>
<|im_start|>assistant



##### call a model to generate a response

In [13]:
interp_toolkit.generate(model, this_prompt, max_new_tokens=50)

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:02<00:00, 17.45it/s]


['<|im_start|>system\nYou are a helpful assistant..<|im_end|>\n<|im_start|>user\nWhat is the capital city of France?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nThe capital city of France is **Paris**. Located in the northern part of the country, it is situated on the banks of the Seine river and has been the political and cultural heart of the nation since the 12th century']

##### call a model and get activations and logits

In [16]:
logits,hidden_states = interp_toolkit.get_activations(model, this_prompt, layer=0, token_pos  = -1, hook_type="residual_pre")

### initial demo for mean difference vectors and projections

In [65]:
stimuli_positive = ["The capital of France is Paris.", "Paris is the capital city of France.", "France's capital is Paris."]
stimuli_negative = ["The capital of France is Berlin.", "Berlin is the capital city of France.", "France's capital is Berlin."]
token_indices_positive = [(1, 3), (2, 4), (0, 2)]  # Example token indices for positive stimuli
token_indices_negative = [(1, 3), (2, 4), (0, 2)]  # Example token indices for negative stimuli
hook_type = 'residual_pre'
layer = 0
system_message = "You are a helpful assistant."

'qwen'

In [80]:
dir = interp_toolkit.extract_activation_vector(model,
                                               model_family = interp_toolkit.MODEL_FAMILY, 
                                               positive_stimuli=stimuli_positive, 
                                               negative_stimuli=stimuli_negative,
                                                token_indices_positive=token_indices_positive, 
                                                token_indices_negative=token_indices_negative, 
                                                hook_type=hook_type, layer=layer,
                                                system_message=system_message)

TypeError: expected Tensor as element 0 in argument 0, but got tuple

In [ ]:
#work in functionality from vsem

def extract_activation_vector(strings_1, strings_2, model):    

    toks_1 = tokenize_instructions_fn(text=strings_1, tokenizer=model.tokenizer)
    toks_2 = tokenize_instructions_fn(text=strings_2, tokenizer=model.tokenizer)

        
    _, toks_1_cache = model.run_with_cache(toks_1, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    _, toks_2_cache = model.run_with_cache(toks_2, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    
    pos = -1
    num_layers = model.cfg.n_layers
    vector_directions_by_layer = {}
    
    for layer in range(num_layers):

        hook_name = ('resid_pre', layer) 
            
        toks_1_activations = toks_1_cache[hook_name]
        toks_2_activations = toks_2_cache[hook_name]

        toks_1_last = toks_1_activations[:, pos, :] 
        toks_2_last = toks_2_activations[:, pos, :]

        toks_1_mean_act = toks_1_last.mean(dim=0)
        toks_2_mean_act = toks_2_last.mean(dim=0)

        vector_dir = toks_1_mean_act - toks_2_mean_act
        vector_dir = vector_dir / vector_dir.norm()
        vector_directions_by_layer[layer] = vector_dir
    
    del toks_1_cache, toks_2_cache
    gc.collect(); torch.cuda.empty_cache()
    
    return vector_directions_by_layer

In [ ]:
#from seong
def calculate_projection(model, tokenize_instructions_fn, items, vector_directions_by_layer):

    #get the number of items to project and layers
    num_items = len(items)
    num_layers = len(vector_directions_by_layer)
    print(num_items)
    print(num_layers)

    #initialize an array to hold the projections
    item_projection = np.zeros((num_items, num_layers))
    
    #tokenize the items all at once and send all to the model to obtain their embeddings
    tokens_batch = tokenize_instructions_fn(text=items, tokenizer=model.tokenizer)    
    _, cache = model.run_with_cache(tokens_batch, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    print(cache.keys())
    print(tokens_batch.shape)
    
    #go through each item embedding and calculate the inner product with the vector direction
   
        
    for layer in range(num_layers):

        # get the item embeddings for the current layer using the cache varaible saved above
        hook_name = ('resid_pre', layer)  #this layer activation name
        this_activation = cache[hook_name]          
        print((this_activation.shape))   #should be batch size (num items), length of each batch, and layer size

        for item in range(num_items):

            pos = -1       
            last_token_activation = this_activation[item, pos, :]   #get last token activation, for this batch item          
        
            last_token_activation_norm = last_token_activation / last_token_activation.norm(dim=0, keepdim=True) #normalize
            item_activation_norm = last_token_activation_norm.detach().cpu().numpy()
            #finally, get the vector direction for the current layer
            this_direction = vector_directions_by_layer[layer].detach().cpu().numpy()
            #now project this item onto the vector direction
            item_projection[item, layer] = np.inner(this_direction, item_activation_norm)
    
    return item_projection

In [ ]:
#from vsem
    def project_stimuli_onto_directions(self, stimuli: list[str], concept_directions: dict,
                                      hook_point: str = "resid_pre", batch_size: int = 1) -> np.ndarray:
        """
        Project stimuli activations onto concept direction vectors.            
        Returns:
            Array of shape (num_stimuli, num_layers) with projection values
        """
        if not stimuli:
            raise ValueError("Stimuli list cannot be empty")
        
        print(f"Projecting {len(stimuli)} stimuli onto concept directions")
        
        num_stimuli = len(stimuli)
        num_layers = len(concept_directions)
        projections = np.zeros((num_stimuli, num_layers))
        
        for batch_start in range(0, num_stimuli, batch_size):
            batch_end = min(batch_start + batch_size, num_stimuli)
            batch_stimuli = stimuli[batch_start:batch_end]
            
            # Tokenize and run forward pass
            batch_tokens = self.tokenize_fn(self.model.tokenizer, batch_stimuli)
            _, cache = self.model.run_with_cache(
                batch_tokens, names_filter=lambda hook_name: hook_point in hook_name
            )
        
            for batch_idx, stimulus_idx in enumerate(range(batch_start, batch_end)):
                for layer in range(num_layers):
                    hook_name = (hook_point, layer)
                    activations = cache[hook_name]
                    
                    # Get activation at last token position
                    activation = activations[batch_idx, -1, :]
                    
                    # Normalize activation
                    activation_norm = activation / activation.norm(dim=0, keepdim=True)
                    
                    # Convert to numpy for projection calculation
                    activation_np = activation_norm.detach().cpu().numpy()
                    direction_np = concept_directions[layer].detach().cpu().numpy()
                    
                    # Compute projection
                    projections[stimulus_idx, layer] = np.inner(direction_np, activation_np)
        
        gc.collect()
        torch.cuda.empty_cache()
        
        print(f"Computed projections with shape: {projections.shape}")
        return projections

In [ ]:
moral_vector_by_layer = extract_activation_vector(morally_positive_adjectives, morally_negative_adjectives,model)

In [ ]:
print(moral_vector_by_layer)

In [ ]:
moral_projections = calculate_projection(
    model=model,
    tokenize_instructions_fn=tokenize_instructions_fn,
    items=stimuli_seong68['item'].tolist(),
    vector_directions_by_layer=moral_vector_by_layer
)

In [33]:
dd = {
    "llama": """<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
{system_prompt}
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user_prompt}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""
}

In [37]:
dd['llama']

'<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\n{system_prompt}\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\n{user_prompt}\n<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n'

###  archive below

In [ ]:
def return_stimuli_seong68(stim_path):
    """
    Returns the list of stimuli from the Seong 68 dataset.
    """
    stim_file = os.path.join(stim_path, 'seong_68.csv')
    stimuli_seong68 = pd.read_csv(stim_file,header=0)
    # stimuli_seong68 = stimuli['item'].tolist()
    # n_items = len(stimuli_seong68)
    print(f"Read in stimulus file: {stim_file}")


    return stimuli_seong68

In [ ]:
stimuli_seong68 = return_stimuli_seong68(stim_path)


In [ ]:
def return_stimuli_dillion(stim_path):
    """
    Returns the list of stimuli from the Dillion dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'dillion_2023.csv')
    stimuli_dillion = pd.read_csv(stim_file,header=0)
    # stimuli_dillion = stimuli['Item'].tolist()
    # n_items = len(stimuli_dillion)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_dillion

In [ ]:
stimuli_dillion = return_stimuli_dillion(stim_path)
print(stimuli_dillion)

In [ ]:
def return_stimuli_moral_positive(stim_path):
    """
    Returns the list of stimuli from the Moral Positive dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'scenarios_moral_seong_positive.csv')
    stimuli = pd.read_csv(stim_file,header=None)
    stimuli_moral_positive =  stimuli.iloc[:,0].tolist()
    # n_items = len(stimuli_moral_positive)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_moral_positive
def return_stimuli_moral_negative(stim_path):
    """
    Returns the list of stimuli from the Moral Negative dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'scenarios_moral_seong_negative.csv')
    stimuli = pd.read_csv(stim_file,header=None)
    stimuli_moral_negative =  stimuli.iloc[:,0].tolist()
    # n_items = len(stimuli_moral_negative)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_moral_negative
    

In [ ]:
scenarios_morally_positive = return_stimuli_moral_positive(stim_path)
scenarios_morally_negative = return_stimuli_moral_negative(stim_path)

In [ ]:
def return_adjectives(stim_path):
    adjfile = os.path.join(stim_path, 'adjectives.json')
    f = open(adjfile, 'r')
    adjectives_dict = json.load(f)
    
    return adjectives_dict

In [ ]:
morally_positive_adjectives = return_adjectives(stim_path)['morality_high']
morally_negative_adjectives = return_adjectives(stim_path)['morality_low']


In [ ]:
morally_positive_adjectives

### archive below

### some basic demos for transfomerlens model interaction

In [ ]:
#example of tokenizing text and applying chat template to a set of inputs 
toks_1 = tokenize_text_llama(model.tokenizer,scenarios_morally_positive)
print(toks_1)
print(toks_1.shape)

In [ ]:
#example of one call to get model activations. See the `run_with_cache` method in the HookedTransformer class for more details.
_, toks_1_cache = model.run_with_cache(toks_1, names_filter=lambda hook_name: 'resid_pre' in hook_name)
print(toks_1_cache.keys())
print(toks_1_cache['blocks.0.hook_resid_pre'].shape) #dimensions are the #batches of input tokens; the size of each batch; and then the layer dimensions